In [4]:
import json
import math
from pathlib import Path
from scipy.stats import norm

# ----------------------------
# Inputs
# ----------------------------

json_paths = [
    Path(r"master_outputs\square\stouffers_jsons\chain_nonzero_psd_scan12.json"),
    Path(r"master_outputs\square\stouffers_jsons\chain_nonzero_psd_scan13.json"),
    Path(r"master_outputs\square\stouffers_jsons\chain_nonzero_psd_scan14.json"),
]

CHAIN_TYPES = {"excitatory", "inhibitory"}

def signed_z_from_p(p, alternative):
    z = norm.isf(p)
    return z if alternative == "greater" else -z

def stouffer_weighted(zs, ws):
    Z = sum(w * z for w, z in zip(ws, zs)) / math.sqrt(sum(w**2 for w in ws))
    p = norm.sf(abs(Z))
    return Z, p

records = []

for path in json_paths:
    with open(path, "r") as f:
        scan = json.load(f)
        for row in scan:
            if row["chain_type"] in CHAIN_TYPES:
                n1 = row["n_shared"]
                n2 = row["n_disjoint"]

                records.append({
                    "chain": row["chain_type"],
                    "z": signed_z_from_p(row["p_value"], row["alternative"]),
                    "n1": n1,
                    "n2": n2
                })

for chain in CHAIN_TYPES:
    subset = [r for r in records if r["chain"] == chain]

    zs = [r["z"] for r in subset]

    w_eff = [
        math.sqrt((r["n1"] * r["n2"]) / (r["n1"] + r["n2"]))
        for r in subset
    ]

    Z_eff, p_eff = stouffer_weighted(zs, w_eff)

    # Total sample size weights
    w_tot = [
        math.sqrt(r["n1"] + r["n2"])
        for r in subset
    ]

    Z_tot, p_tot = stouffer_weighted(zs, w_tot)

    # ----------------------------
    # Report (matches your style)
    # ----------------------------

    print("\n~~~~~~~~~~~~~~~~~~~~~~~~~~~~~\n")
    print(f"{chain.upper()} — Stouffer's with direction, using effective sample size weights:")
    print("w_i = sqrt[(n1 * n2) / (n1 + n2)]")
    print(f"Z_meta: {Z_eff}")
    print(f"p_meta: {p_eff}")

    print("\n~~~~~~~~~~~~~~~~~~~~~~~~~~~~~\n")
    print(f"{chain.upper()} — Stouffer's with direction, using total sample size weights:")
    print("w_i = sqrt(n1 + n2)")
    print(f"Combined Z-score: {Z_tot}")
    print(f"Combined p-value: {p_tot}")


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

EXCITATORY — Stouffer's with direction, using effective sample size weights:
w_i = sqrt[(n1 * n2) / (n1 + n2)]
Z_meta: 0.8411240437749503
p_meta: 0.20013922333156603

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

EXCITATORY — Stouffer's with direction, using total sample size weights:
w_i = sqrt(n1 + n2)
Combined Z-score: 0.9982765916452359
Combined p-value: 0.15907262764254682

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

INHIBITORY — Stouffer's with direction, using effective sample size weights:
w_i = sqrt[(n1 * n2) / (n1 + n2)]
Z_meta: -2.7996287639147797
p_meta: 0.002558070359376888

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

INHIBITORY — Stouffer's with direction, using total sample size weights:
w_i = sqrt(n1 + n2)
Combined Z-score: -2.7675796469258884
Combined p-value: 0.0028237122105662883
